# Avaliação da API de Score de Crédito no Google Colab

Este notebook chama a API publicada no Cloud Run e mede a qualidade das respostas com dados **rotulados**.

1. Lê a chave da API de forma segura (Secrets do Colab ou `getpass`); a chave nunca fica no notebook.
2. Carrega o `train.csv` do repositório e usa **agosto**, o mês de validação out-of-time do treino.
3. Converte cada linha bruta para o contrato da API (`POST /v1/score/lote`, até 100 clientes por chamada).
4. Respeita o throttling (30 requisições/min por chave) e trata `429` com `Retry-After`.
5. Compara o score previsto com o rótulo real: acurácia, F1 macro, recall de `Poor`, matriz de confusão e latência.
6. Mostra os erros padronizados da API (401, 422, 413).

**Antes de rodar:** se você recebeu o **código de convite**, gere sua chave na célula "Não tem chave?".
Se já tem uma chave, guarde-a nos Secrets do Colab 🔑 como `QF_API_KEY` e ative o acesso do notebook.

In [ ]:
# Configuração
API_URL = "https://api-score-credito-czkhbhag2q-rj.a.run.app"   # URL do Cloud Run (sem barra no final)
DADOS_URL = "https://raw.githubusercontent.com/rfc890305/mlop_score_credito/main/data/raw/train.csv"
MES_AVALIACAO = "August"   # mês de validação usado no treino
AMOSTRA = 2000             # nº de clientes avaliados (None = mês inteiro, cerca de 12 mil; leva uns 5 min)
TAMANHO_LOTE = 100         # máximo aceito por /v1/score/lote
INTERVALO_S = 2.1          # 60 s / 30 requisições por minuto, com folga
SEMENTE = 42

### Não tem chave? Gere a sua aqui (uma vez só)

Quem recebeu o **código de convite** do responsável pelo projeto gera a própria chave pela API
(`POST /v1/chaves`). Preencha nome e e-mail no formulário e rode a célula; o código é pedido com
digitação oculta. A chave aparece **uma única vez**: guarde-a nos Secrets do Colab 🔑 como `QF_API_KEY`
para não precisar gerar outra. Ela vale por 30 dias. Se você já tem chave, pule esta célula.

In [ ]:
# Autocadastro: gera uma chave pessoal com o código de convite (pule se já tem chave)
NOME = ""   #@param {type:"string"}
EMAIL = ""  #@param {type:"string"}
import os, getpass, requests
URL_CADASTRO = "https://api-score-credito-czkhbhag2q-rj.a.run.app/v1/chaves"
codigo = getpass.getpass("Código de convite: ").strip().strip("\"'")
r = requests.post(URL_CADASTRO, json={"nome": NOME, "email": EMAIL, "codigo_convite": codigo}, timeout=60)
if r.status_code == 201:
    emitida = r.json()
    os.environ["QF_API_KEY"] = emitida["chave"]   # a próxima célula já usa esta chave
    print("Chave gerada (copie e guarde nos Secrets do Colab como QF_API_KEY; ela não será exibida de novo):")
    print(emitida["chave"])
    print(f"Válida até {emitida['expira_em'][:10]}. Identificador para revogação: {emitida['prefixo']}...")
else:
    erro = r.json().get("erro", {})
    print(f"HTTP {r.status_code}: {erro.get('mensagem', r.text)}")
    if r.status_code == 422:
        print("Confira o nome (mínimo 2 letras) e o e-mail.")

In [ ]:
# Chave da API: Secrets do Colab; fora do Colab, variável de ambiente ou digitação oculta
import os, getpass
# 1) chave gerada agora pelo autocadastro; 2) Secrets do Colab; 3) digitação oculta
API_KEY = os.environ.get("QF_API_KEY")
if not API_KEY:
    try:
        from google.colab import userdata
        API_KEY = userdata.get("QF_API_KEY")
    except Exception:
        API_KEY = getpass.getpass("Chave da API (X-API-Key): ")
API_KEY = (API_KEY or "").strip().strip("\"'")   # remove espaços, quebras de linha e aspas coladas por engano
HEADERS = {"X-API-Key": API_KEY, "Content-Type": "application/json"}
# Mostra só o tamanho e as pontas, para conferir com o Cloud Shell sem expor a chave inteira
print(f"Chave carregada: {len(API_KEY)} caracteres ({API_KEY[:4]}...{API_KEY[-4:]})")
if len(API_KEY) != 43:
    print("Atenção: a chave gerada pelo configurar_gcp.sh tem 43 caracteres. Confira com "
          "`gcloud secrets versions access latest --secret=qf-api-keys | wc -c` no Cloud Shell.")

## 1. A API está no ar? Qual modelo está servindo?

In [ ]:
import requests
saude = requests.get(f"{API_URL}/health", timeout=30)
print(saude.status_code, saude.json())
modelo = requests.get(f"{API_URL}/v1/modelo", headers=HEADERS, timeout=30)
print(modelo.status_code, modelo.json())
if modelo.status_code == 401:
    print("\nChave recusada. Causas comuns: (1) caracteres a mais ou a menos na cópia (o Cloud Shell cola o prompt"
          " logo após a chave); (2) chave criada depois do último deploy: o Cloud Run só lê o segredo ao criar uma"
          " revisão. Rode: gcloud run services update api-score-credito --region=southamerica-east1"
          " --update-secrets=QF_API_KEYS=qf-api-keys:latest")

## 2. Dados rotulados e conversão para o contrato da API

O CSV bruto tem sujeira (`"28_"`, `"_"`, idade `-500`, histórico como texto `"22 Years and 3 Months"`).
A API valida tudo com rigor e recusa valores fora das faixas (422). Aqui, a limpeza segue as mesmas regras
de `src/credit_score/processamento.py`, e linhas com algum campo inválido ficam de fora da avaliação.

In [ ]:
import numpy as np, pandas as pd

FAIXAS = {
    "Age": (14, 100), "Annual_Income": (0, 250_000), "Monthly_Inhand_Salary": (0, 25_000),
    "Num_Bank_Accounts": (0, 20), "Num_Credit_Card": (0, 15), "Interest_Rate": (0, 40),
    "Num_of_Loan": (0, 15), "Delay_from_due_date": (-10, 100), "Num_of_Delayed_Payment": (0, 40),
    "Changed_Credit_Limit": (-50, 50), "Num_Credit_Inquiries": (0, 30), "Outstanding_Debt": (0, 10_000),
    "Credit_Utilization_Ratio": (0, 100), "Credit_History_Age": (0, 600), "Total_EMI_per_month": (0, 5_000),
    "Amount_invested_monthly": (0, 9_999), "Monthly_Balance": (-5_000, 5_000),
}
INTEIROS = ["Age", "Num_Bank_Accounts", "Num_Credit_Card", "Num_of_Loan", "Delay_from_due_date",
            "Num_of_Delayed_Payment", "Num_Credit_Inquiries", "Credit_History_Age"]
CATEGORIAS = {
    "Credit_Mix": ["Bad", "Standard", "Good"],
    "Payment_of_Min_Amount": ["Yes", "No", "NM"],
    "Payment_Behaviour": [f"{g}_spent_{v}_value_payments" for g in ("Low", "High") for v in ("Small", "Medium", "Large")],
}

def numero(s):
    return pd.to_numeric(s.astype("string").str.strip().str.strip("_"), errors="coerce").astype("float64")

def historico_meses(s):
    x = s.astype("string").str.extract(r"(\d+)\s*Years?\s*and\s*(\d+)\s*Months?")
    return x[0].astype("float64") * 12 + x[1].astype("float64")

bruto = pd.read_csv(DADOS_URL, low_memory=False)
mes = bruto[bruto["Month"] == MES_AVALIACAO].copy()

dados = pd.DataFrame({"cliente_id": mes["Customer_ID"].astype(str)}, index=mes.index)
for col, (mn, mx) in FAIXAS.items():
    v = historico_meses(mes[col]) if col == "Credit_History_Age" else numero(mes[col])
    dados[col] = v.where(v.between(mn, mx))
for col, validos in CATEGORIAS.items():
    dados[col] = mes[col].where(mes[col].isin(validos))
dados["rotulo"] = mes["Credit_Score"]

validos = dados.dropna()
print(f"{MES_AVALIACAO}: {len(mes)} linhas; {len(validos)} válidas para a API "
      f"({len(mes) - len(validos)} descartadas por campo inválido ou ausente)")
if AMOSTRA and AMOSTRA < len(validos):
    validos = validos.sample(AMOSTRA, random_state=SEMENTE)
print("Distribuição dos rótulos avaliados:")
print(validos["rotulo"].value_counts(normalize=True).round(3))

In [ ]:
def para_payload(df):
    registros = []
    for linha in df.drop(columns="rotulo").to_dict("records"):
        for c in INTEIROS:
            linha[c] = int(linha[c])
        registros.append(linha)
    return registros

print(para_payload(validos.head(1))[0])

## 3. Chamadas em lote, respeitando o throttling

In [ ]:
import time

def chamar_lote(clientes, tentativas=5):
    for _ in range(tentativas):
        inicio = time.perf_counter()
        r = requests.post(f"{API_URL}/v1/score/lote", headers=HEADERS, json={"clientes": clientes}, timeout=60)
        latencia = time.perf_counter() - inicio
        if r.status_code == 429:
            espera = int(r.headers.get("Retry-After", 10))
            print(f"429 limite_excedido: aguardando {espera} s")
            time.sleep(espera)
            continue
        if r.status_code >= 500:
            time.sleep(5)
            continue
        r.raise_for_status()   # 401/403/422 aqui indicam problema de chave ou de payload
        return r.json(), latencia
    raise RuntimeError(f"Falhou após {tentativas} tentativas: {r.status_code} {r.text[:300]}")

resultados, latencias, versoes = [], [], set()
payload = para_payload(validos)
for i in range(0, len(payload), TAMANHO_LOTE):
    resposta, lat = chamar_lote(payload[i:i + TAMANHO_LOTE])
    resultados.extend(resposta["resultados"])
    latencias.append(lat)
    versoes.add(resposta["modelo"]["versao"])
    print(f"lote {i // TAMANHO_LOTE + 1}: {resposta['total']} clientes em {lat:.2f} s")
    time.sleep(INTERVALO_S)

previsto = pd.DataFrame({
    "score": [r["score"] for r in resultados],
    **{f"prob_{c}": [r["probabilidades"][c] for r in resultados] for c in ("Poor", "Standard", "Good")},
}, index=validos.index)
avaliacao = validos[["cliente_id", "rotulo"]].join(previsto)
print("Versões do modelo que responderam:", versoes)
avaliacao.head()

## 4. Qualidade das respostas

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, recall_score, classification_report, confusion_matrix, roc_auc_score

CLASSES = ["Poor", "Standard", "Good"]
y, p = avaliacao["rotulo"], avaliacao["score"]
ORDEM = sorted(CLASSES)   # roc_auc_score exige as colunas em ordem alfabética
probs = avaliacao[[f"prob_{c}" for c in ORDEM]].values
probs = probs / probs.sum(axis=1, keepdims=True)   # a API arredonda as probabilidades
metricas = {
    "acuracia": accuracy_score(y, p),
    "f1_macro": f1_score(y, p, average="macro"),
    "recall_Poor": recall_score(y, p, labels=["Poor"], average="macro"),
    "roc_auc_ovr": roc_auc_score(y, probs, multi_class="ovr", labels=ORDEM),
}
print(pd.Series(metricas).round(4))
print()
print(classification_report(y, p, labels=CLASSES, digits=3))

# Mesmos limites usados na promoção do modelo (config/config.yaml)
print("F1 macro >= 0,60:", "OK" if metricas["f1_macro"] >= 0.60 else "ABAIXO")
print("Recall Poor >= 0,60:", "OK" if metricas["recall_Poor"] >= 0.60 else "ABAIXO")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, eixos = plt.subplots(1, 2, figsize=(12, 4.5))
ConfusionMatrixDisplay(confusion_matrix(y, p, labels=CLASSES), display_labels=CLASSES).plot(ax=eixos[0], cmap="Blues", colorbar=False)
eixos[0].set_title("Matriz de confusão (real × previsto pela API)")
eixos[1].hist(np.array(latencias) * 1000, bins=15)
eixos[1].set_title(f"Latência por lote de {TAMANHO_LOTE} (mediana {np.median(latencias)*1000:.0f} ms)")
eixos[1].set_xlabel("ms")
plt.tight_layout(); plt.show()

print("Latência por lote (s): p50=%.2f  p95=%.2f  máx=%.2f" % tuple(np.percentile(latencias, [50, 95, 100])))

In [ ]:
# Onde o modelo mais erra: confiança média nos acertos e nos erros
avaliacao["confianca"] = avaliacao[[f"prob_{c}" for c in CLASSES]].max(axis=1)
avaliacao["acertou"] = avaliacao["rotulo"] == avaliacao["score"]
print(avaliacao.groupby("acertou")["confianca"].describe().round(3))
avaliacao[~avaliacao["acertou"]].sort_values("confianca", ascending=False).head(10)

**Como ler o resultado:** agosto é o mês de validação usado no treino. Por isso, o F1 macro obtido aqui deve ficar
próximo do registrado no MLflow (cerca de 0,68 para o modelo padrão). A diferença vem da amostragem e das linhas descartadas:
em agosto, cerca de 60% das linhas têm algum campo sujo ou fora da faixa e não passam na validação da API. Como sobram
os registros mais limpos, o F1 aqui tende a ficar um pouco acima do offline (em teste local com o modelo v4, deu 0,717
contra 0,701 no MLflow). Uma queda grande indicaria divergência entre treino e produção
(training-serving skew) ou outro modelo servindo, o que se confere em `GET /v1/modelo`.

## 5. Respostas de erro da API (contrato documentado em `docs/API.md`)

In [ ]:
exemplo = para_payload(validos.head(1))[0]
casos = {
    "sem chave (401)": dict(headers={"Content-Type": "application/json"}, json=exemplo),
    "chave inválida (401)": dict(headers={"X-API-Key": "chave-errada"}, json=exemplo),
    "idade fora da faixa (422)": dict(headers=HEADERS, json={**exemplo, "Age": 150}),
    "campo desconhecido (422)": dict(headers=HEADERS, json={**exemplo, "campo_extra": 1}),
}
for nome, kw in casos.items():
    r = requests.post(f"{API_URL}/v1/score", timeout=30, **kw)
    print(f"{nome}: HTTP {r.status_code} -> {r.json()['erro']['codigo']}  (X-Request-ID {r.headers.get('X-Request-ID')})")
    time.sleep(INTERVALO_S)

r = requests.post(f"{API_URL}/v1/score/lote", headers=HEADERS, json={"clientes": [exemplo] * 101}, timeout=30)
print(f"lote com 101 clientes: HTTP {r.status_code} -> {r.json()['erro']['codigo']}")

## 6. (Opcional) Salvar o resultado

In [ ]:
avaliacao.to_csv("avaliacao_api.csv", index=False)
pd.Series(metricas).to_json("metricas_api.json")
print("Arquivos salvos: avaliacao_api.csv e metricas_api.json (painel de arquivos à esquerda no Colab)")